# OMOP CDM Gold Layer Overview

This notebook provides an overview of the **standalone** Gold layer that transforms FHIR Silver tables into OMOP Common Data Model (CDM) v5.4 format.

## Standalone Architecture

This pipeline operates **independently** from the FHIR Bronze/Silver ingestion pipeline. It uses Delta Streaming to automatically detect and process new records from Silver tables.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  FHIR INGESTION PIPELINE (Separate)                                         │
│  Bronze → Silver Tables                                                     │
│  Patient | Encounter | Condition | Procedure | Observation | ...            │
└────────────────────────────────┬────────────────────────────────────────────┘
                                 │
                                 │ Delta Streaming (Automatic CDC)
                                 │ No direct pipeline dependency
                                 ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│  OMOP GOLD PIPELINE (Standalone)                                            │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  Foundation Tables (Phase 1 - Parallel)                              │   │
│  │  • person           ← Patient                                        │   │
│  │  • care_site        ← Organization                                   │   │
│  │  • provider         ← Practitioner                                   │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
│                                    ↓                                        │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  Visit Table (Phase 2)                                               │   │
│  │  • visit_occurrence ← Encounter                                      │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
│                                    ↓                                        │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  Clinical Tables (Phase 3 - Parallel)                                │   │
│  │  • condition_occurrence  ← Condition                                 │   │
│  │  • drug_exposure         ← MedicationRequest                         │   │
│  │  • procedure_occurrence  ← Procedure                                 │   │
│  │  • measurement           ← Observation (labs/vitals)                 │   │
│  │  • observation           ← Observation (other)                       │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────────────────┘
```

## Key Benefits of Standalone Design

1. **Decoupled Execution**: OMOP pipeline runs on its own schedule
2. **Automatic CDC**: Streaming tables detect new Silver records automatically
3. **Fault Tolerance**: FHIR pipeline failures don't block OMOP processing
4. **Independent Scaling**: Can adjust OMOP refresh frequency separately
5. **Easier Testing**: Can test OMOP transformations without FHIR ingestion

## Deployment Options

| Option | Job | Trigger | Use Case |
|--------|-----|---------|----------|
| Scheduled | `omop_gold_pipeline` | Every 10 min | Near-realtime analytics |
| Manual | Run notebooks | On-demand | Testing, ad-hoc refresh |
| Continuous | Modify job trigger | Continuous | True realtime (higher cost) |

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';  -- FHIR Silver tables
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';      -- OMOP Gold tables

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

SELECT 
  catalog_use AS catalog,
  silver_schema AS fhir_silver,
  gold_schema AS omop_gold;

In [ ]:
-- Create Gold schema if not exists
DECLARE OR REPLACE VARIABLE create_schema_stmt STRING;
SET VARIABLE create_schema_stmt = 'CREATE SCHEMA IF NOT EXISTS ' || catalog_use || '.' || gold_schema;
EXECUTE IMMEDIATE create_schema_stmt;

In [ ]:
-- Verify Silver tables are available (source for OMOP)
DECLARE OR REPLACE VARIABLE check_silver_stmt STRING;

SET VARIABLE check_silver_stmt = "
SELECT 
  table_name AS silver_table,
  CASE 
    WHEN table_name = 'patient' THEN 'person'
    WHEN table_name = 'encounter' THEN 'visit_occurrence'
    WHEN table_name = 'condition' THEN 'condition_occurrence'
    WHEN table_name = 'medicationrequest' THEN 'drug_exposure'
    WHEN table_name = 'procedure' THEN 'procedure_occurrence'
    WHEN table_name = 'observation' THEN 'measurement + observation'
    WHEN table_name = 'practitioner' THEN 'provider'
    WHEN table_name = 'organization' THEN 'care_site'
    ELSE '(not mapped)'
  END AS omop_target,
  table_type
FROM " || catalog_use || ".information_schema.tables
WHERE table_schema = '" || silver_schema || "'
ORDER BY table_name
";

EXECUTE IMMEDIATE check_silver_stmt;

In [ ]:
-- Check existing OMOP Gold tables
DECLARE OR REPLACE VARIABLE check_gold_stmt STRING;

SET VARIABLE check_gold_stmt = "
SELECT 
  table_name AS omop_table,
  table_type
FROM " || catalog_use || ".information_schema.tables
WHERE table_schema = '" || gold_schema || "'
ORDER BY table_name
";

EXECUTE IMMEDIATE check_gold_stmt;

## FHIR to OMOP Mapping Summary

| FHIR Resource | OMOP Table | Key Mappings |
|---------------|------------|---------------|
| Patient | person | gender→8507/8532, birthDate→year/month/day_of_birth |
| Encounter | visit_occurrence | class→visit_concept_id (9201/9202/9203) |
| Condition | condition_occurrence | code→condition_concept_id (SNOMED/ICD) |
| MedicationRequest | drug_exposure | medication→drug_concept_id (RxNorm) |
| Procedure | procedure_occurrence | code→procedure_concept_id (SNOMED/CPT) |
| Observation (lab/vitals) | measurement | code→measurement_concept_id (LOINC) |
| Observation (other) | observation | code→observation_concept_id |
| Practitioner | provider | NPI, name, specialty |
| Organization | care_site | name, type→place_of_service |

## OMOP Standard Concept IDs Reference

### Gender Concepts
| Gender | concept_id |
|--------|------------|
| Male | 8507 |
| Female | 8532 |
| Unknown | 0 |

### Visit Concepts
| Visit Type | concept_id |
|------------|------------|
| Inpatient Visit | 9201 |
| Outpatient Visit | 9202 |
| Emergency Room Visit | 9203 |
| Long Term Care Visit | 42898160 |
| ER + Inpatient Visit | 262 |

### Type Concepts
| Type | concept_id | Use Case |
|------|------------|----------|
| EHR | 32817 | *_type_concept_id (general) |
| Lab result | 32856 | measurement_type_concept_id |
| Prescription written | 38000177 | drug_type_concept_id |

## References

- [OMOP CDM v5.4 Documentation](https://ohdsi.github.io/CommonDataModel/cdm54.html)
- [HL7 FHIR to OMOP IG](https://build.fhir.org/ig/HL7/fhir-omop-ig/)
- [OHDSI Athena Vocabulary](https://athena.ohdsi.org/)